# Twitter Entity Sentiment Analysis with BERT

## Problem Statement
We are given tweets and an associated entity (e.g., a company or product).  
For each tweet, we must classify the sentiment **towards that entity** into one of three categories:  
- **Positive**
- **Negative**
- **Neutral** (where `Irrelevant` is mapped to `Neutral`)

**Dataset source:**  
[Twitter Entity Sentiment Analysis](https://www.kaggle.com/datasets/jp797498e/twitter-entity-sentiment-analysis/data) on Kaggle.

## Result
We fine‑tune a pre‑trained `bert‑base‑uncased` model for this task.  
After training, the model achieves a **validation accuracy of 97.90%** with excellent precision and recall across all classes (see final evaluation for the full classification report).

---

## 1. Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## 2. Load and Explore the Data

In [2]:
train_path = 'twitter_training.csv'   # Update path if needed
val_path   = 'twitter_validation.csv'

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)

print(f"Training samples: {train_df.shape[0]}")
print(f"Validation samples: {val_df.shape[0]}")
print("\nSample rows from training data:")
display(train_df.head())

# Distribution of sentiments
print("\nTraining sentiment distribution:")
print(train_df['sentiment'].value_counts())
print("\nValidation sentiment distribution:")
print(val_df['sentiment'].value_counts())

Training samples: 74682
Validation samples: 1000

Sample rows from training data:


,tweet_id,entity,sentiment,tweet_content
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...



Training sentiment distribution:
sentiment
Negative      22542
Positive      20832
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64

Validation sentiment distribution:
sentiment
Neutral       285
Positive      277
Negative      266
Irrelevant    172
Name: count, dtype: int64


## 3. Preprocessing
We map `Irrelevant` to `Neutral` and encode the labels.  
Then we create a combined input text: `entity [SEP] tweet_content`.

In [3]:
# Label mapping
sentiment_to_label = {
    'Positive': 0,
    'Negative': 1,
    'Neutral':  2,
    'Irrelevant': 2   # Irrelevant → Neutral
}
label_to_sentiment = {v: k for k, v in sentiment_to_label.items() if k != 'Irrelevant'}  # {0:'Positive', 1:'Negative', 2:'Neutral'}

# Apply mapping
train_df['label'] = train_df['sentiment'].map(sentiment_to_label)
val_df['label']   = val_df['sentiment'].map(sentiment_to_label)

# Create combined text
def create_input_text(row):
    return f"{row['entity']} [SEP] {row['tweet_content']}"

train_df['input_text'] = train_df.apply(create_input_text, axis=1)
val_df['input_text']   = val_df.apply(create_input_text, axis=1)

print("Example input text:")
print(train_df['input_text'].iloc[0])

Example input text:
Borderlands [SEP] im getting on borderlands and i will murder you all ,


## 4. Tokenization and Dataset Preparation
We use `BertTokenizer` with a maximum length of 64 tokens (tweets are usually short).  
The `TweetDataset` class returns tensors required for training.

In [4]:
# Tokenizer and model
model_name = '../bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)

MAX_LEN = 64   # suitable for tweet-like texts

class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [5]:
# Create datasets and loaders
train_texts = train_df['input_text'].tolist()
train_labels = train_df['label'].tolist()
val_texts = val_df['input_text'].tolist()
val_labels = val_df['label'].tolist()

train_dataset = TweetDataset(train_texts, train_labels, tokenizer)
val_dataset   = TweetDataset(val_texts, val_labels, tokenizer)

BATCH_SIZE = 16   # adjust based on GPU memory

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Validation batches: {len(val_loader)}')

Train batches: 4668
Validation batches: 63


## 5. Model and Training Setup
We load a pre‑trained BERT model with a classification head for 3 classes.  
The optimizer is `AdamW` (from `torch.optim`) and we use a linear learning rate scheduler.

In [6]:
model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
).to(device)

# Optimizer and scheduler
EPOCHS = 3
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, eps=1e-8)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

loss_fn = nn.CrossEntropyLoss().to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ../bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 6. Training and Evaluation Functions
The training loop iterates over batches, performs forward/backward passes, and updates weights.  
The evaluation function computes loss and accuracy.

In [7]:
def train_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    model.eval()
    preds, true_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)
            total_loss += loss.item()

            logits = outputs.logits
            _, predicted = torch.max(logits, dim=1)
            preds.extend(predicted.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    accuracy = accuracy_score(true_labels, preds)
    return total_loss / len(dataloader), accuracy, preds, true_labels

## 7. Training Loop
We train for `EPOCHS` iterations, saving the best model based on validation accuracy.

In [8]:
best_accuracy = 0
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, device)

    print(f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Val accuracy: {val_acc:.4f}")

    if val_acc > best_accuracy:
        best_accuracy = val_acc
        torch.save(model.state_dict(), 'best_model.bin')
        print("Best model saved!")

print(f"\nBest validation accuracy: {best_accuracy:.4f}")


Epoch 1/3


Evaluating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 63/63 [00:11<00:00,  5.48it/s]


Train loss: 0.5709 | Val loss: 0.1636 | Val accuracy: 0.9530
Best model saved!

Epoch 2/3


Evaluating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 63/63 [00:12<00:00,  5.07it/s]


Train loss: 0.2133 | Val loss: 0.0930 | Val accuracy: 0.9770
Best model saved!

Epoch 3/3


Evaluating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 63/63 [00:12<00:00,  5.06it/s]


Train loss: 0.1098 | Val loss: 0.1151 | Val accuracy: 0.9790
Best model saved!

Best validation accuracy: 0.9790


## 8. Final Evaluation
Load the best model and generate a classification report.

In [9]:
model.load_state_dict(torch.load('best_model.bin'))
val_loss, val_acc, all_preds, all_labels = evaluate(model, val_loader, device)
print(f"Validation accuracy: {val_acc:.4f}")

print("\nClassification Report:")
print(classification_report(
    all_labels, all_preds,
    target_names=['Positive', 'Negative', 'Neutral']
))

Evaluating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 63/63 [00:12<00:00,  5.08it/s]

Validation accuracy: 0.9790

Classification Report:
              precision    recall  f1-score   support

    Positive       0.96      0.99      0.97       277
    Negative       0.99      0.98      0.99       266
     Neutral       0.98      0.97      0.98       457

    accuracy                           0.98      1000
   macro avg       0.98      0.98      0.98      1000
weighted avg       0.98      0.98      0.98      1000



## 9. Conclusion
This notebook demonstrates a complete pipeline for entity‑level tweet sentiment analysis using a fine‑tuned BERT model.  
Key steps:
- Mapping `Irrelevant` to `Neutral`.
- Creating a joint `entity [SEP] tweet` input.
- Training with `AdamW` and a linear warm‑up scheduler.
- Evaluating using accuracy and classification report.

The trained model can be further improved by:
- Using a larger BERT variant (`bert‑large‑uncased`).
- Hyperparameter tuning (learning rate, batch size).
- Adding data augmentation.
- Training for more epochs.